# AIMO Interpretability — Temporal Uncertainty E1/E2 (Kaggle)

This notebook runs the first research gate for our AIMO Interpretability project:

\[
E1\;(\text{official 14D}) \quad \text{vs.} \quad E2\;(\text{Temporal Uncertainty v1, 66D})
\]

**Goal:** determine whether temporal uncertainty dynamics add a real robustness-prediction signal under the official grouped-CV protocol.

### Recommended Kaggle settings
- Accelerator: **2×T4** if available; L4/A100 is also fine.
- Internet: **ON** (needed for GitHub + Hugging Face downloads unless you attach local model/data assets).
- Start with `MODE="smoke"`, then switch to `MODE="pilot"` only after the smoke test passes.
- Do **not** use Kaggle for the later 27B/120B, hidden-state, or counterfactual stages; those are candidates for official H200 compute.

This notebook uses branch `research/temporal-uncertainty-v1` from `luxury221/getting-started`.


In [ ]:
# ---------------------------
# Experiment configuration
# ---------------------------
MODE = "smoke"  # "smoke", "pilot", or "full"

REPO_URL = "https://github.com/luxury221/getting-started.git"
BRANCH = "research/temporal-uncertainty-v1"
MODEL_ID = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"

PRESETS = {
    # Pure engineering smoke test. CV may be skipped if there are too few groups/classes.
    "smoke": dict(max_samples=8, max_new_tokens=128, batch_size=1, quick_cv=True),
    # First meaningful E1/E2 pilot. Increase to 96/128 if class-group coverage is insufficient.
    "pilot": dict(max_samples=64, max_new_tokens=256, batch_size=1, quick_cv=True),
    # Full public training-set extraction. On Kaggle this may be slow; official compute is preferred.
    "full": dict(max_samples=None, max_new_tokens=4096, batch_size=1, quick_cv=False),
}
CFG = PRESETS[MODE]
CFG


In [ ]:
# ---------------------------
# GPU / runtime diagnostics
# ---------------------------
import os, sys, subprocess, platform, json, pathlib, shutil

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1024**3:.1f} GiB")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for the 8B feature-extraction smoke test.")


In [ ]:
# ---------------------------
# Clone the research branch
# ---------------------------
WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "getting-started"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)],
    check=True,
)
print("Repo:", REPO)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], check=True)


In [ ]:
# ---------------------------
# Install competition-compatible Python deps
# Keep Kaggle's CUDA-enabled torch instead of replacing it.
# ---------------------------
pkgs = [
    "accelerate==1.13.0",
    "huggingface-hub==1.22.0",
    "joblib==1.5.3",
    "pandas==3.0.3",
    "safetensors==0.8.0",
    "scikit-learn==1.8.0",
    "tokenizers==0.22.2",
    "transformers==5.13.0",
    "pyarrow>=17",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Do not import these before the installation cell if you want exact versions.
import transformers, sklearn, pandas as pd, numpy as np, joblib, accelerate
print("transformers", transformers.__version__)
print("sklearn", sklearn.__version__)
print("pandas", pd.__version__)
print("accelerate", accelerate.__version__)


In [ ]:
# ---------------------------
# Unit tests for Temporal Uncertainty v1
# ---------------------------
test_cmd = [
    sys.executable, "-m", "unittest", "discover",
    str(REPO / "solutions/uncertainty-profiling/tests"),
    "-p", "test_temporal_metrics.py",
]
subprocess.run(test_cmd, cwd=REPO, check=True)
print("Temporal feature unit tests: PASS")


In [ ]:
# ---------------------------
# Feature extraction
# ---------------------------
OUT = WORK / "aimo_outputs"
OUT.mkdir(exist_ok=True)
HF_CACHE = WORK / "hf-cache"
HF_CACHE.mkdir(exist_ok=True)

feature_path = OUT / f"temporal_v1_{MODE}.parquet"

cmd = [
    sys.executable,
    str(REPO / "solutions/uncertainty-profiling/scripts/compute_temporal_features.py"),
    "--feature-model-id", MODEL_ID,
    "--cache-dir", str(HF_CACHE),
    "--max-new-tokens", str(CFG["max_new_tokens"]),
    "--batch-size", str(CFG["batch_size"]),
    "--output", str(feature_path),
    "--overwrite",
]
if CFG["max_samples"] is not None:
    cmd += ["--max-samples", str(CFG["max_samples"])]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)
print("Saved:", feature_path)


In [ ]:
# ---------------------------
# Inspect extracted feature cache
# ---------------------------
import pandas as pd
df = pd.read_parquet(feature_path)

print("Rows:", len(df))
print("Unique problems:", df["original_problem"].nunique())
print("Columns:", len(df.columns))
print("\nRobustness label counts:")
print(df["model_is_robust"].value_counts(dropna=False))

temporal_cols = [c for c in df.columns if c.startswith("temporal_")]
official_cols = [c for c in df.columns if c.startswith("generation_")]

print("\nOfficial generation columns:", len(official_cols))
print("Temporal columns:", len(temporal_cols))
display(df.head(3))


In [ ]:
# ---------------------------
# Choose a leakage-safe number of grouped-CV folds
# ---------------------------
group_labels = (
    df[["original_problem", "model_is_robust"]]
    .drop_duplicates("original_problem")
)
class_group_counts = group_labels["model_is_robust"].value_counts()

if len(class_group_counts) < 2:
    SAFE_SPLITS = 0
else:
    SAFE_SPLITS = min(5, int(class_group_counts.min()))

print("Unique-group class counts:")
print(class_group_counts)
print("SAFE_SPLITS =", SAFE_SPLITS)

if SAFE_SPLITS < 2:
    print(
        "\nSmoke extraction passed, but this subset cannot support stratified grouped CV. "
        "Switch MODE='pilot' and rerun the notebook."
    )


In [ ]:
# ---------------------------
# E0 / E1 / E2 grouped-CV ablation
# ---------------------------
results_path = OUT / f"temporal_ablation_{MODE}.json"

if SAFE_SPLITS >= 2:
    cmd = [
        sys.executable,
        str(REPO / "solutions/uncertainty-profiling/scripts/run_temporal_ablation.py"),
        "--feature-data-path", str(feature_path),
        "--n-splits", str(SAFE_SPLITS),
        "--seed", "42",
        "--results-path", str(results_path),
    ]
    if CFG["quick_cv"]:
        cmd.append("--quick")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)
else:
    print("Skipping CV because SAFE_SPLITS < 2.")


In [ ]:
# ---------------------------
# Compact E1/E2 result table + decision gate
# ---------------------------
if results_path.exists():
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    rows = []
    e0 = results["E0_majority"]
    rows.append({
        "experiment": "E0_majority",
        "features": 0,
        "balanced_accuracy": e0["balanced_accuracy"],
        "accuracy": e0["accuracy"],
        "mae": None,
        "family": "majority",
    })

    for exp, payload in results["experiments"].items():
        m = payload["best_cv_metrics"]
        rows.append({
            "experiment": exp,
            "features": payload["feature_count"],
            "balanced_accuracy": m["balanced_accuracy"],
            "accuracy": m["accuracy"],
            "mae": m["mae"],
            "family": payload["best_family"],
        })

    result_df = pd.DataFrame(rows)
    display(result_df.sort_values("experiment"))

    e1 = results["experiments"]["E1_official_14d"]["best_cv_metrics"]
    e2 = results["experiments"]["E2D_full_temporal_v1"]["best_cv_metrics"]
    delta_ba = e2["balanced_accuracy"] - e1["balanced_accuracy"]
    delta_acc = e2["accuracy"] - e1["accuracy"]

    print(f"E2D - E1 balanced accuracy: {delta_ba:+.4f}")
    print(f"E2D - E1 accuracy:          {delta_acc:+.4f}")

    if MODE == "smoke":
        print(
            "\nSMOKE MODE: treat these metrics only as a pipeline check. "
            "Switch to MODE='pilot' before making a research decision."
        )
    elif delta_ba >= 0.03 and delta_acc >= -0.02:
        print("\nDECISION: GO — Temporal signal is strong enough to justify full replication / Phase II planning.")
    elif delta_ba > 0:
        print("\nDECISION: PROMISING — rerun on the full public set before entering Representation Dynamics.")
    else:
        print("\nDECISION: HOLD — do not spend H200 budget on Phase II yet; inspect feature families and failure modes.")
else:
    print("No CV results yet. Use MODE='pilot' if the smoke subset lacked enough groups.")


## What to keep after the run

Kaggle will preserve files under `/kaggle/working/aimo_outputs/` in the notebook output when you **Save Version**.

The key files are:

- `temporal_v1_<mode>.parquet` — extracted 66D features.
- `temporal_ablation_<mode>.json` — E0/E1/E2 grouped-CV results.

### Research gate

Do **not** move to hidden-state dynamics just because one tiny split improves.

Our first meaningful gate is:

1. `pilot` passes with a positive E2D−E1 balanced-accuracy delta;
2. the gain is not caused only by one accidental fold;
3. then run the full public dataset;
4. only after full-data confirmation do we prepare the official H200 compute proposal for Representation Dynamics + Counterfactual Stability.
